# FabricOps AI DQ + Governance — Clean Notebook Prototype

This notebook keeps the workflow clean and notebook-first:

1. Profile the source data.
2. AI drafts column business context.
3. Human approves business context once.
4. DQ AI uses profile + approved business context.
5. Human approves DQ rules.
6. Governance AI uses profile + approved business context.
7. Human approves governance classifications.
8. Metadata is stored in separate tables joined by deterministic keys.

Metadata tables stay separate:

- `METADATA_COLUMN_CONTEXT`
- `METADATA_DQ_RULES`
- `METADATA_COLUMN_GOVERNANCE`

Column-level tables share `metadata_column_key` based on environment, dataset, table, and column.


In [1]:
from __future__ import annotations

import ast
import copy
import hashlib
import html
import json
import re
import uuid
from datetime import datetime, timezone
from typing import Any

import pandas as pd
import ipywidgets as widgets
from IPython.display import display as ipy_display
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
)

# Fabric AI Functions extension used by df.ai.generate_response(...)
from synapse.ml.spark.aifunc.DataFrameExtensions import AIFunctions


StatementMeta(, 808313c9-f002-4cdf-801e-04546705732f, 3, Submitted, Waiting, Running, True)

In [ ]:
def _now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def _resolve_action_by(action_by: str | None = None) -> str:
    if action_by:
        return str(action_by)
    try:
        import notebookutils.runtime as nb_runtime
        context = getattr(nb_runtime, "context", None)
        if isinstance(context, dict):
            return context.get("userName") or context.get("userId") or "unknown"
        get_method = getattr(context, "get", None)
        if callable(get_method):
            return get_method("userName") or get_method("userId") or "unknown"
    except Exception:
        pass
    return "unknown"


def _key_part(value: Any) -> str:
    return "" if value is None else str(value).strip().lower()


def _sha256_key(*parts: Any) -> str:
    return hashlib.sha256("|".join(_key_part(p) for p in parts).encode("utf-8")).hexdigest()


def build_metadata_table_key(environment_name, dataset_name, table_name) -> str:
    """Build deterministic table metadata key from environment, dataset, and table."""
    return _sha256_key(environment_name, dataset_name, table_name)


def build_metadata_column_key(environment_name, dataset_name, table_name, column_name) -> str:
    """Build deterministic column metadata key from environment, dataset, table, and column."""
    return _sha256_key(environment_name, dataset_name, table_name, column_name)


def build_dq_rule_key(environment_name, dataset_name, table_name, rule_id) -> str:
    """Build deterministic DQ rule key from environment, dataset, table, and rule id."""
    return _sha256_key(environment_name, dataset_name, table_name, rule_id)


def _extract_columns_from_profile(profile_df) -> list[str]:
    cols = set(profile_df.columns)
    if "column_name" in cols:
        return [str(r["column_name"]) for r in profile_df.select("column_name").collect()]
    if "COLUMN_NAME" in cols:
        return [str(r["COLUMN_NAME"]) for r in profile_df.select("COLUMN_NAME").collect()]
    raise ValueError("profile_df must contain column_name or COLUMN_NAME.")


def normalise_records_by_column(records) -> dict[str, dict]:
    """Accept dict keyed by column or list of dicts with column_name/COLUMN_NAME."""
    if not records:
        return {}
    if isinstance(records, dict):
        return records
    if isinstance(records, list):
        out = {}
        for item in records:
            if not isinstance(item, dict):
                continue
            col = item.get("column_name") or item.get("COLUMN_NAME")
            if col:
                out[str(col)] = item
        return out
    raise ValueError("records must be a dict, a list of dicts, or None.")


def column_context_rows_for_spark(
    spark,
    profile_columns: list[str],
    column_contexts=None,
):
    """Build a Spark DataFrame with approved business context per profiled column."""
    context_map = normalise_records_by_column(column_contexts)
    rows = []
    for col in profile_columns:
        item = context_map.get(str(col), {})
        if not isinstance(item, dict):
            item = {}
        rows.append(
            {
                "column_name": str(col),
                "business_context": str(item.get("business_context", "") or ""),
                "business_context_notes": str(item.get("notes", "") or item.get("business_context_notes", "") or ""),
            }
        )
    return spark.createDataFrame(rows)


def _esc(value: Any) -> str:
    return html.escape(str(value))


StatementMeta(, , -1, Waiting, , Waiting, True)

## 1. Business context first

Business context is captured once and reused by DQ and governance prompts.


In [ ]:
COLUMN_BUSINESS_CONTEXT_FROM_WIDGET = []

BUSINESS_CONTEXT_PROMPT = """
You are helping a data analyst document column-level business context for a data product.

Your job:
Infer a short, practical business meaning for this column based on the table name, column name, data type, profile statistics, sample values, and overall table context.

Return ONLY a valid Python dictionary with this exact shape:
{
  "column_name": "<column_name>",
  "business_context": "<short business meaning of the column>",
  "notes": "<short caveat, assumption, or empty string>"
}

Guidelines:
- Be concise.
- Do not invent sensitive facts.
- If the meaning is uncertain, say so in notes.
- Focus on what the column likely represents to a business user.
- Do not classify personal data here. This widget is only for business meaning.

Table name: {table_name}
Overall table context: {table_context}

Column profile:
Column name: {column_name}
Data type: {data_type}
Row count: {row_count}
Null count: {null_count}
Distinct count: {distinct_count}
Min value: {min_value}
Max value: {max_value}
Observed values sample: {observed_values_sample}
"""


def _prepare_business_context_profile_input(profile_df, table_name: str, table_context: str = ""):
    """Normalise profile rows for Fabric AI business context suggestion."""
    cols = set(profile_df.columns)

    if {"column_name", "data_type"}.issubset(cols):
        out = profile_df
    elif {"COLUMN_NAME", "DATA_TYPE"}.issubset(cols):
        out = profile_df.select(
            F.lit(table_name).alias("table_name"),
            F.col("COLUMN_NAME").alias("column_name"),
            F.col("DATA_TYPE").alias("data_type"),
            F.col("ROW_COUNT").alias("row_count") if "ROW_COUNT" in cols else F.lit(None).alias("row_count"),
            F.col("NULL_COUNT").alias("null_count") if "NULL_COUNT" in cols else F.lit(None).alias("null_count"),
            F.col("DISTINCT_COUNT").alias("distinct_count") if "DISTINCT_COUNT" in cols else F.lit(None).alias("distinct_count"),
            F.col("MIN_VALUE").alias("min_value") if "MIN_VALUE" in cols else F.lit(None).alias("min_value"),
            F.col("MAX_VALUE").alias("max_value") if "MAX_VALUE" in cols else F.lit(None).alias("max_value"),
        )
    else:
        raise ValueError("profile_df must contain either column_name/data_type or COLUMN_NAME/DATA_TYPE.")

    required_defaults = {
        "table_name": F.lit(table_name),
        "row_count": F.lit(None),
        "null_count": F.lit(None),
        "distinct_count": F.lit(None),
        "min_value": F.lit(None),
        "max_value": F.lit(None),
        "observed_values_sample": F.lit(""),
        "table_context": F.lit(table_context),
    }

    out_cols = set(out.columns)
    for col_name, default_expr in required_defaults.items():
        if col_name not in out_cols:
            out = out.withColumn(col_name, default_expr)

    return out.select(
        "table_name",
        "column_name",
        "data_type",
        "row_count",
        "null_count",
        "distinct_count",
        "min_value",
        "max_value",
        "observed_values_sample",
        "table_context",
    )


def draft_business_context(
    profile_df,
    table_name: str,
    table_context: str = "",
    prompt_template: str = BUSINESS_CONTEXT_PROMPT,
    output_col: str = "ai_response",
):
    """Use Fabric AI Functions to generate first-pass column business context."""
    prepared = _prepare_business_context_profile_input(profile_df, table_name, table_context)
    return prepared.ai.generate_response(prompt=prompt_template, output_col=output_col)


def _parse_ai_dict_response(text: str | None) -> dict | None:
    raw = str(text or "").strip()
    if not raw:
        return None
    raw = raw.replace("```python", "").replace("```json", "").replace("```", "").strip()
    match = re.search(r"(\{.*\})", raw, flags=re.DOTALL)
    payload = match.group(1) if match else raw
    try:
        parsed = json.loads(payload)
    except Exception:
        try:
            parsed = ast.literal_eval(payload)
        except Exception:
            return None
    return parsed if isinstance(parsed, dict) else None


def _extract_column_business_context_suggestions(response_df, response_col: str = "ai_response") -> list[dict]:
    """Parse Fabric AI business-context responses into list[dict]."""
    suggestions = []
    for row in response_df.select(response_col).collect():
        parsed = _parse_ai_dict_response(row[response_col])
        if not parsed:
            continue
        column_name = parsed.get("column_name") or parsed.get("COLUMN_NAME")
        if not column_name:
            continue
        suggestions.append(
            {
                "column_name": str(column_name),
                "business_context": str(parsed.get("business_context") or parsed.get("context") or ""),
                "notes": str(parsed.get("notes") or parsed.get("note") or ""),
                "source": "ai_business_context_suggestion",
            }
        )
    return suggestions


def review_business_context(
    profile_df=None,
    columns=None,
    table_name: str | None = None,
    environment_name: str | None = None,
    dataset_name: str | None = None,
    ai_suggested_contexts=None,
    existing_contexts=None,
):
    """Fabric widget for approving reusable column-level business context."""
    global COLUMN_BUSINESS_CONTEXT_FROM_WIDGET
    COLUMN_BUSINESS_CONTEXT_FROM_WIDGET.clear()

    if columns is None:
        if profile_df is None:
            raise ValueError("Provide either profile_df or columns.")
        columns = _extract_columns_from_profile(profile_df)
    columns = [str(c) for c in columns]

    ai_map = normalise_records_by_column(ai_suggested_contexts)
    existing_map = normalise_records_by_column(existing_contexts)
    state = {"index": 0, "history": []}

    title = widgets.HTML(value=f"<h4 style='margin:0;'>Column business context capture: {_esc(table_name or '')}</h4>")
    progress_label = widgets.HTML()
    column_summary = widgets.HTML()
    ai_suggestion_box = widgets.HTML()

    column_name_box = widgets.Text(description="Column", disabled=True, layout=widgets.Layout(width="650px"))
    business_context_box = widgets.Textarea(
        description="Business context",
        placeholder="What does this column mean in business terms? How is it used?",
        layout=widgets.Layout(width="780px", height="120px"),
    )
    notes_box = widgets.Textarea(
        description="Notes",
        placeholder="Optional notes, assumptions, known caveats, or review comments.",
        layout=widgets.Layout(width="780px", height="90px"),
    )
    save_button = widgets.Button(description="Approve / Save Context", button_style="success", layout=widgets.Layout(width="240px"))
    skip_button = widgets.Button(description="Skip Column", button_style="warning", layout=widgets.Layout(width="180px"))
    undo_button = widgets.Button(description="Undo Last", layout=widgets.Layout(width="160px"))
    status = widgets.HTML()
    form_box = widgets.VBox([column_name_box, ai_suggestion_box, business_context_box, notes_box, widgets.HBox([save_button, skip_button, undo_button])])

    def current_column():
        idx = state["index"]
        return None if idx >= len(columns) else columns[idx]

    def load_current_column():
        col = current_column()
        progress_label.value = f"<b>Progress:</b> {state['index']} / {len(columns)} &nbsp; | &nbsp; <b>Saved:</b> {len(COLUMN_BUSINESS_CONTEXT_FROM_WIDGET)}"

        if col is None:
            column_summary.value = f"""
            <div style="border:1px solid #d1e7dd;background:#f0fff4;padding:14px;border-radius:8px;margin-top:8px;">
                <div style="font-size:18px;font-weight:700;color:#166534;margin-bottom:6px;">✅ Business context capture complete</div>
                <div><b>Table:</b> {_esc(table_name or '')}</div>
                <div><b>Columns reviewed:</b> {state['index']} / {len(columns)}</div>
                <div><b>Saved context rows:</b> {len(COLUMN_BUSINESS_CONTEXT_FROM_WIDGET)}</div>
                <div style="margin-top:10px;color:#444;">Next step: use <b>COLUMN_BUSINESS_CONTEXT_FROM_WIDGET</b> in DQ and governance AI prompts.</div>
            </div>"""
            form_box.layout.display = "none"
            status.value = "<span style='color:#166534;font-weight:600;'>Column business context is ready for AI suggestion workflows.</span>"
            return

        ai_context = ai_map.get(col, {}) if isinstance(ai_map.get(col, {}), dict) else {}
        existing = existing_map.get(col, {}) if isinstance(existing_map.get(col, {}), dict) else {}
        suggested_context = ai_context.get("business_context", "")
        suggested_notes = ai_context.get("notes", "")

        column_summary.value = f"""
        <div style="border:1px solid #e5e7eb;background:#fafafa;padding:12px;border-radius:8px;margin-top:8px;">
            <div style="font-weight:700;margin-bottom:8px;">Column {state['index'] + 1} of {len(columns)}</div>
            <div><b>Column:</b> <code>{_esc(col)}</code></div>
            <div style="margin-top:8px;color:#444;">AI gives a first pass. User edits and approves the final business meaning.</div>
        </div>"""
        column_name_box.value = col

        if suggested_context or suggested_notes:
            ai_suggestion_box.value = f"""
            <div style="border:1px solid #dbeafe;background:#eff6ff;padding:10px;border-radius:8px;margin-top:6px;margin-bottom:6px;">
                <div style="font-weight:700;color:#1d4ed8;">AI first-pass business context</div>
                <div><b>Context:</b> {_esc(suggested_context)}</div>
                <div><b>Notes:</b> {_esc(suggested_notes)}</div>
                <div style="margin-top:6px;color:#444;">This has been prefilled below. Edit before approving.</div>
            </div>"""
        else:
            ai_suggestion_box.value = """
            <div style="border:1px solid #e5e7eb;background:#fafafa;padding:10px;border-radius:8px;margin-top:6px;margin-bottom:6px;">
                <div style="font-weight:700;">AI first-pass business context</div>
                <div style="color:#666;">No AI suggestion supplied for this column.</div>
            </div>"""

        business_context_box.value = str(existing.get("business_context") or suggested_context or "")
        notes_box.value = str(existing.get("notes") or suggested_notes or "")
        form_box.layout.display = ""
        status.value = ""

    def save_clicked(_):
        col = current_column()
        if col is None:
            status.value = "<span style='color:red'>No current column to save.</span>"
            return
        ai_context = ai_map.get(col, {}) if isinstance(ai_map.get(col, {}), dict) else {}
        row = {
            "environment_name": environment_name,
            "dataset_name": dataset_name,
            "table_name": table_name,
            "column_name": col,
            "metadata_table_key": build_metadata_table_key(environment_name, dataset_name, table_name),
            "metadata_column_key": build_metadata_column_key(environment_name, dataset_name, table_name, col),
            "business_context": business_context_box.value.strip(),
            "notes": notes_box.value.strip(),
            "ai_suggested_business_context": ai_context.get("business_context"),
            "ai_suggested_notes": ai_context.get("notes"),
            "status": "approved",
            "action_by": _resolve_action_by(),
            "action_ts": _now_utc_iso(),
            "source": "column_business_context_widget",
        }
        COLUMN_BUSINESS_CONTEXT_FROM_WIDGET.append(row)
        state["history"].append({"action": "save", "saved_count": 1})
        state["index"] += 1
        load_current_column()

    def skip_clicked(_):
        if current_column() is None:
            status.value = "<span style='color:red'>No current column to skip.</span>"
            return
        state["history"].append({"action": "skip", "saved_count": 0})
        state["index"] += 1
        load_current_column()

    def undo_clicked(_):
        if not state["history"]:
            status.value = "<span style='color:orange'>Nothing to undo.</span>"
            return
        last = state["history"].pop()
        if last["saved_count"]:
            del COLUMN_BUSINESS_CONTEXT_FROM_WIDGET[-last["saved_count"]:]
        state["index"] = max(0, state["index"] - 1)
        load_current_column()

    save_button.on_click(save_clicked)
    skip_button.on_click(skip_clicked)
    undo_button.on_click(undo_clicked)

    ui = widgets.VBox([title, progress_label, column_summary, form_box, status], layout=widgets.Layout(border="1px solid #ddd", padding="12px", width="860px"))
    load_current_column()
    ipy_display(ui)
    return COLUMN_BUSINESS_CONTEXT_FROM_WIDGET


StatementMeta(, , -1, Waiting, , Waiting, True)

## 2. Data quality functions

DQ AI suggestions now use approved column business context when available.


In [ ]:
AI_SUGGESTABLE_DQ_RULE_TYPES = {
    "not_null",
    "unique_key",
    "accepted_values",
    "value_range",
    "regex_format",
}
DQ_RULE_SUGGESTION_PROMPT_TEMPLATE = """
You are helping draft candidate FabricOps data quality rules for a pipeline contract notebook.

These suggestions are advisory only. A human must approve them before enforcement.

Use only these FabricOps rule_type values:
- not_null
- unique_key
- accepted_values
- value_range
- regex_format

Use the approved column business context heavily. The technical profile alone is not enough.
Do not suggest unsupported rule types.
Do not return Great Expectations, Deequ, DQX, SQL, or pseudocode syntax.

Return only a Python dictionary named DQ_RULES using this shape:

DQ_RULES = {
    "{table_name}": [
        {
            "rule_id": "lower_snake_case_rule_id",
            "rule_type": "one_supported_rule_type",
            "columns": ["column_name"],
            "severity": "error_or_warning",
            "description": "Plain business explanation."
        }
    ]
}

For accepted_values, include allowed_values.
For value_range, include lower_bound and/or upper_bound.
Do not use min_value or max_value.
For regex_format, include regex_pattern.

Example value_range rule:
{
    "rule_id": "event_count_value_range",
    "rule_type": "value_range",
    "columns": ["event_count"],
    "severity": "warning",
    "description": "Event count should be zero or positive.",
    "lower_bound": 0
}

Table name:
{table_name}

Column profile row:
Column name: {column_name}
Data type: {data_type}
Row count: {row_count}
Null count: {null_count}
Null percent: {null_percent}
Distinct count: {distinct_count}
Distinct percent: {distinct_percent}
Minimum observed value: {min_value}
Maximum observed value: {max_value}
Observed values sample: {observed_values_sample}

Approved business context:
{business_context}

Business context notes:
{business_context_notes}
"""


def profile_dataframe_for_dq(df, table_name: str, sample_value_limit: int = 20):
    """Create one profile row per source column for AI-assisted DQ and governance workflows."""
    row_count = df.count()
    profile_rows = []

    for column_name, data_type in df.dtypes:
        null_count = df.filter(F.col(column_name).isNull()).count()
        distinct_count = df.select(column_name).distinct().count()
        min_value = None
        max_value = None

        if data_type in {"int", "bigint", "double", "float", "date", "timestamp"} or data_type.startswith("decimal"):
            min_max = df.select(
                F.min(F.col(column_name)).cast("string").alias("min_value"),
                F.max(F.col(column_name)).cast("string").alias("max_value"),
            ).collect()[0]
            min_value = min_max["min_value"]
            max_value = min_max["max_value"]

        observed_values = [
            str(row[column_name])
            for row in df.select(column_name).where(F.col(column_name).isNotNull()).distinct().limit(sample_value_limit).collect()
        ]

        profile_rows.append(
            {
                "table_name": table_name,
                "column_name": column_name,
                "data_type": data_type,
                "row_count": int(row_count),
                "null_count": int(null_count),
                "null_percent": round((null_count / row_count) * 100, 4) if row_count else 0.0,
                "distinct_count": int(distinct_count),
                "distinct_percent": round((distinct_count / row_count) * 100, 4) if row_count else 0.0,
                "min_value": min_value,
                "max_value": max_value,
                "observed_values_sample": ", ".join(observed_values),
                "profile_timestamp": _now_utc_iso(),
            }
        )

    return df.sparkSession.createDataFrame(profile_rows)


def prepare_dq_profile_input(profile_df, table_name: str, column_contexts=None):
    """Join approved column business context into profile rows before DQ AI suggestion."""
    cols = set(profile_df.columns)
    if {"column_name", "data_type"}.issubset(cols):
        out = profile_df
    elif {"COLUMN_NAME", "DATA_TYPE"}.issubset(cols):
        out = profile_df.select(
            F.lit(table_name).alias("table_name"),
            F.col("COLUMN_NAME").alias("column_name"),
            F.col("DATA_TYPE").alias("data_type"),
            F.col("ROW_COUNT").alias("row_count") if "ROW_COUNT" in cols else F.lit(None).alias("row_count"),
            F.col("NULL_COUNT").alias("null_count") if "NULL_COUNT" in cols else F.lit(None).alias("null_count"),
            F.col("NULL_PERCENT").alias("null_percent") if "NULL_PERCENT" in cols else F.lit(None).alias("null_percent"),
            F.col("DISTINCT_COUNT").alias("distinct_count") if "DISTINCT_COUNT" in cols else F.lit(None).alias("distinct_count"),
            F.col("DISTINCT_PERCENT").alias("distinct_percent") if "DISTINCT_PERCENT" in cols else F.lit(None).alias("distinct_percent"),
            F.col("MIN_VALUE").alias("min_value") if "MIN_VALUE" in cols else F.lit(None).alias("min_value"),
            F.col("MAX_VALUE").alias("max_value") if "MAX_VALUE" in cols else F.lit(None).alias("max_value"),
        )
    else:
        raise ValueError("profile_df must contain either column_name/data_type or COLUMN_NAME/DATA_TYPE.")

    required_defaults = {
        "table_name": F.lit(table_name),
        "row_count": F.lit(None),
        "null_count": F.lit(None),
        "null_percent": F.lit(None),
        "distinct_count": F.lit(None),
        "distinct_percent": F.lit(None),
        "min_value": F.lit(None),
        "max_value": F.lit(None),
        "observed_values_sample": F.lit(""),
    }
    out_cols = set(out.columns)
    for col_name, default_expr in required_defaults.items():
        if col_name not in out_cols:
            out = out.withColumn(col_name, default_expr)

    out = out.select(
        "table_name",
        "column_name",
        "data_type",
        "row_count",
        "null_count",
        "null_percent",
        "distinct_count",
        "distinct_percent",
        "min_value",
        "max_value",
        "observed_values_sample",
    )

    profile_columns = [r["column_name"] for r in out.select("column_name").collect()]
    context_df = column_context_rows_for_spark(profile_df.sparkSession, profile_columns, column_contexts)
    return (
        out.join(context_df, on="column_name", how="left")
        .select(
            "table_name",
            "column_name",
            "data_type",
            "row_count",
            "null_count",
            "null_percent",
            "distinct_count",
            "distinct_percent",
            "min_value",
            "max_value",
            "observed_values_sample",
            F.coalesce(F.col("business_context"), F.lit("")).alias("business_context"),
            F.coalesce(F.col("business_context_notes"), F.lit("")).alias("business_context_notes"),
        )
    )


def suggest_dq_rules_with_fabric_ai(
    profile_df,
    table_name: str,
    column_contexts=None,
    prompt_template: str = DQ_RULE_SUGGESTION_PROMPT_TEMPLATE,
    output_col: str = "response",
):
    """Use Fabric AI to suggest candidate DQ rules using profile + approved business context."""
    prepared = prepare_dq_profile_input(profile_df, table_name=table_name, column_contexts=column_contexts)
    return prepared.ai.generate_response(prompt=prompt_template, output_col=output_col)


def parse_dq_rules_dict_from_text(text: str) -> dict[str, list[dict[str, Any]]]:
    """Parse text shaped like `DQ_RULES = {...}` into a Python dictionary."""
    if text is None:
        return {}
    cleaned = str(text).strip().replace("```python", "").replace("```json", "").replace("```", "").strip()
    match = re.search(r"DQ_RULES\s*=\s*(\{.*\})", cleaned, flags=re.DOTALL)
    payload = match.group(1) if match else cleaned
    try:
        parsed = ast.literal_eval(payload)
    except Exception:
        return {}
    return parsed if isinstance(parsed, dict) else {}


def attach_rule_metadata_keys(rules, environment_name=None, dataset_name=None, table_name=None):
    """Attach deterministic table/rule/column keys to candidate or approved rules."""
    out = []
    for rule in rules:
        r = copy.deepcopy(rule)
        columns = r.get("columns", []) or []
        r["environment_name"] = environment_name
        r["dataset_name"] = dataset_name
        r["table_name"] = table_name
        r["metadata_table_key"] = build_metadata_table_key(environment_name, dataset_name, table_name)
        r["rule_key"] = build_dq_rule_key(environment_name, dataset_name, table_name, r.get("rule_id"))
        r["column_keys"] = [build_metadata_column_key(environment_name, dataset_name, table_name, c) for c in columns]
        if len(columns) == 1:
            r["metadata_column_key"] = r["column_keys"][0]
        out.append(r)
    return out


def extract_candidate_rules_from_responses(
    response_df,
    table_name: str,
    response_col: str = "response",
    environment_name: str | None = None,
    dataset_name: str | None = None,
):
    """Extract, de-duplicate, and key candidate rules from Fabric AI response rows."""
    rows = response_df.select(response_col).collect()
    candidate_rules = []

    for row in rows:
        parsed = parse_dq_rules_dict_from_text(row[response_col])
        candidate_rules.extend(parsed.get(table_name, []))

    deduped = {}
    for rule in candidate_rules:
        rule_id = rule.get("rule_id")
        if rule_id:
            deduped[rule_id] = rule

    return attach_rule_metadata_keys(list(deduped.values()), environment_name, dataset_name, table_name)


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
def build_dq_rules_metadata_df(
    spark,
    table_name: str,
    approved_rules: list[dict],
    environment_name: str | None = None,
    dataset_name: str | None = None,
    action_by: str | None = None,
    rule_source: str = "ai_widget_approval",
    action_reason: str = "Approved after human review.",
):
    """Build append-only metadata rows for approved DQ rules."""
    action_ts = _now_utc_iso()
    rows = []
    resolved_action_by = _resolve_action_by(action_by)

    for rule in approved_rules:
        columns = rule.get("columns", []) or []
        rule_id = str(rule["rule_id"])
        env = rule.get("environment_name", environment_name)
        dataset = rule.get("dataset_name", dataset_name)
        table = rule.get("table_name", table_name)
        keyed_rule = attach_rule_metadata_keys([rule], env, dataset, table)[0]

        rows.append(
            {
                "environment_name": env,
                "dataset_name": dataset,
                "table_name": table,
                "metadata_table_key": keyed_rule["metadata_table_key"],
                "rule_id": rule_id,
                "rule_type": str(rule["rule_type"]),
                "columns": ",".join(columns),
                "column_keys": json.dumps(keyed_rule.get("column_keys", [])),
                "rule_key": keyed_rule["rule_key"],
                "is_active": True,
                "action_type": "approved",
                "action_by": resolved_action_by,
                "action_ts": action_ts,
                "action_reason": action_reason,
                "rule_source": rule_source,
                "rule_json": json.dumps(keyed_rule),
            }
        )
    return spark.createDataFrame(rows)


def build_dq_rule_deactivation_metadata_df(
    spark,
    table_name: str,
    deactivations: list[dict],
    environment_name: str | None = None,
    dataset_name: str | None = None,
    action_by: str | None = None,
    rule_source: str = "rule_deactivation_widget",
):
    """Build append-only inactive metadata rows for rules selected for deactivation."""
    action_ts = _now_utc_iso()
    rows = []
    resolved_action_by = _resolve_action_by(action_by)

    for item in deactivations:
        rule = item["rule"]
        action_reason = str(item["action_reason"]).strip()
        if not action_reason:
            raise ValueError(f"Deactivation reason is required for rule '{rule['rule_id']}'.")

        columns = rule.get("columns", []) or []
        env = rule.get("environment_name", environment_name)
        dataset = rule.get("dataset_name", dataset_name)
        table = rule.get("table_name", table_name)
        keyed_rule = attach_rule_metadata_keys([rule], env, dataset, table)[0]

        rows.append(
            {
                "environment_name": env,
                "dataset_name": dataset,
                "table_name": table,
                "metadata_table_key": keyed_rule["metadata_table_key"],
                "rule_id": str(rule["rule_id"]),
                "rule_type": str(rule["rule_type"]),
                "columns": ",".join(columns),
                "column_keys": json.dumps(keyed_rule.get("column_keys", [])),
                "rule_key": keyed_rule["rule_key"],
                "is_active": False,
                "action_type": "deactivated",
                "action_by": resolved_action_by,
                "action_ts": action_ts,
                "action_reason": action_reason,
                "rule_source": rule_source,
                "rule_json": json.dumps(keyed_rule),
            }
        )
    return spark.createDataFrame(rows)


def load_latest_active_dq_rules_from_metadata(metadata_df, table_name: str, environment_name: str | None = None, dataset_name: str | None = None):
    """Resolve latest active approved DQ rules from append-only metadata history."""
    df = metadata_df.filter(F.col("table_name") == table_name)
    if "environment_name" in metadata_df.columns and environment_name is not None:
        df = df.filter(F.col("environment_name") == environment_name)
    if "dataset_name" in metadata_df.columns and dataset_name is not None:
        df = df.filter(F.col("dataset_name") == dataset_name)

    window = Window.partitionBy("rule_key").orderBy(F.col("action_ts").desc())
    latest = df.withColumn("_rn", F.row_number().over(window)).filter(F.col("_rn") == 1).drop("_rn")
    rows = latest.filter(F.col("is_active") == True).select("rule_json").collect()
    return [json.loads(row["rule_json"]) for row in rows]


StatementMeta(, , -1, Waiting, , Waiting, True)

## 3. Data quality review widgets


In [ ]:
import json
import copy
import html
import ipywidgets as widgets
from IPython.display import display as ipy_display

APPROVED_RULES_FROM_WIDGET = []
REJECTED_RULES_FROM_WIDGET = []


DQ_SYSTEM_RULE_KEYS = {
    "rule_id",
    "rule_type",
    "columns",
    "severity",
    "description",
    "environment_name",
    "dataset_name",
    "table_name",
    "metadata_table_key",
    "metadata_column_key",
    "metadata_column_keys",
    "rule_key",
    "column_keys",
    "business_context",
    "business_context_notes",
    "ai_response",
    "source",
    "status",
    "action_by",
    "action_ts",
    "created_by",
    "created_ts",
    "updated_by",
    "updated_ts",
    "is_active",
    "action_type",
    "action_reason",
    "rule_source",
    "rule_json",
}

DQ_NESTED_PARAM_KEYS = {
    "params",
    "parameters",
    "rule_config",
    "config",
}

DQ_USER_EDITABLE_RULE_KEYS = {
    "allowed_values",
    "lower_bound",
    "upper_bound",
    "regex_pattern",
    "mapping",
    "reference_column",
    "reference_values",
    "condition_column",
    "condition_value",
    "expected_column",
    "expected_value",
}

DQ_RULE_PARAM_ALIASES = {
    "min_value": "lower_bound",
    "minimum_value": "lower_bound",
    "lower_limit": "lower_bound",
    "minimum": "lower_bound",
    "max_value": "upper_bound",
    "maximum_value": "upper_bound",
    "upper_limit": "upper_bound",
    "maximum": "upper_bound",
    "values": "allowed_values",
    "accepted_values": "allowed_values",
    "valid_values": "allowed_values",
    "pattern": "regex_pattern",
    "regex": "regex_pattern",
}


def review_dq_rules(candidate_rules, table_name: str, column_contexts=None):
    """
    Grouped batch review widget for AI suggested DQ rules.

    Business context is pulled from column_contexts when available and shown
    as read-only evidence. Rule config only exposes editable rule parameters.
    """

    global APPROVED_RULES_FROM_WIDGET, REJECTED_RULES_FROM_WIDGET

    APPROVED_RULES_FROM_WIDGET.clear()
    REJECTED_RULES_FROM_WIDGET.clear()

    def esc(value):
        return html.escape(str(value))

    def get_rule_type(rule):
        return str(rule.get("rule_type", "unknown"))

    def get_rule_columns(rule):
        cols = rule.get("columns", [])
        if isinstance(cols, list):
            return [str(c) for c in cols]
        if cols is None:
            return []
        return [str(cols)]

    def get_column_label(rule):
        cols = get_rule_columns(rule)
        return ", ".join(cols) if cols else "(no column)"

    def normalize_param_key(key):
        return DQ_RULE_PARAM_ALIASES.get(str(key), str(key))

    def normalise_contexts_by_column(records):
        if not records:
            return {}

        if isinstance(records, dict):
            return records

        if isinstance(records, list):
            out = {}
            for item in records:
                if not isinstance(item, dict):
                    continue
                col = item.get("column_name") or item.get("COLUMN_NAME")
                if col:
                    out[str(col)] = item
            return out

        raise ValueError("column_contexts must be a dict, list of dicts, or None.")

    def enrich_rules_with_column_context(rules, column_contexts):
        context_map = normalise_contexts_by_column(column_contexts)
        enriched = []

        for rule in rules:
            edited = copy.deepcopy(rule)
            cols = get_rule_columns(edited)

            contexts = []
            notes = []

            for col in cols:
                context = context_map.get(str(col), {})
                if not isinstance(context, dict):
                    continue

                business_context = context.get("business_context", "")
                business_notes = context.get("notes", "") or context.get("business_context_notes", "")

                if business_context:
                    contexts.append(f"{col}: {business_context}")

                if business_notes:
                    notes.append(f"{col}: {business_notes}")

            if contexts:
                edited["business_context"] = "\n".join(contexts)

            if notes:
                edited["business_context_notes"] = "\n".join(notes)

            enriched.append(edited)

        return enriched

    def collect_visible_rule_params(rule):
        """
        Return only business editable rule params.

        System metadata, keys, context, action fields, source/status,
        and unknown keys are intentionally hidden.
        """
        params = {}

        def consider_dict(d):
            if not isinstance(d, dict):
                return

            for raw_key, value in d.items():
                key = normalize_param_key(raw_key)

                if key in DQ_SYSTEM_RULE_KEYS:
                    continue

                if key in DQ_NESTED_PARAM_KEYS:
                    if isinstance(value, dict):
                        consider_dict(value)
                    continue

                if key not in DQ_USER_EDITABLE_RULE_KEYS:
                    continue

                params[key] = value

        consider_dict(rule)
        return params

    def clean_user_params(raw_params):
        """
        Clean user edited JSON before applying it back to the rule.

        Protects against stale widgets where system metadata is still
        sitting inside the textarea.
        """
        if not isinstance(raw_params, dict):
            return {}

        cleaned = {}

        for raw_key, value in raw_params.items():
            key = normalize_param_key(raw_key)

            if key in DQ_SYSTEM_RULE_KEYS:
                continue

            if key in DQ_NESTED_PARAM_KEYS:
                if isinstance(value, dict):
                    nested = clean_user_params(value)
                    cleaned.update(nested)
                continue

            if key not in DQ_USER_EDITABLE_RULE_KEYS:
                continue

            if value == "<mixed values across columns>":
                continue

            cleaned[key] = value

        return cleaned

    def normalize_rule_params(rule):
        """
        Copy accepted aliases into canonical parameter names.
        Keeps system metadata untouched.
        """
        out = copy.deepcopy(rule)
        visible_params = collect_visible_rule_params(out)

        for key, value in visible_params.items():
            out[key] = value

        return out

    candidate_rules = enrich_rules_with_column_context(
        candidate_rules,
        column_contexts=column_contexts,
    )

    candidate_rules = [normalize_rule_params(r) for r in candidate_rules]

    def group_rules_by_type(rules):
        grouped = {}

        for rule in rules:
            rule_type = get_rule_type(rule)
            grouped.setdefault(rule_type, []).append(rule)

        return [
            {"rule_type": rule_type, "rules": rules_for_type}
            for rule_type, rules_for_type in grouped.items()
        ]

    def build_common_params_from_rules(rules):
        params = {}

        for rule in rules:
            visible = collect_visible_rule_params(rule)

            for key, value in visible.items():
                if key not in params:
                    params[key] = value
                elif params[key] != value:
                    params[key] = "<mixed values across columns>"

        return params

    def build_business_context_html(rules):
        rows = []

        for rule in rules:
            column_label = get_column_label(rule)
            business_context = rule.get("business_context", "") or ""
            business_context_notes = rule.get("business_context_notes", "") or ""

            rows.append(
                f"""
                <tr>
                    <td style="padding:4px 8px; vertical-align:top;"><code>{esc(column_label)}</code></td>
                    <td style="padding:4px 8px; vertical-align:top;">
                        {esc(business_context) if business_context else "<span style='color:#777;'>No context supplied</span>"}
                    </td>
                    <td style="padding:4px 8px; vertical-align:top;">
                        {esc(business_context_notes) if business_context_notes else ""}
                    </td>
                </tr>
                """
            )

        return f"""
        <details open style="margin-top:8px; margin-bottom:8px;">
            <summary style="cursor:pointer; font-weight:700;">Approved business context used by AI</summary>
            <table style="border-collapse:collapse; margin-top:8px; width:100%;">
                <thead>
                    <tr>
                        <th style="text-align:left; padding:4px 8px;">Column</th>
                        <th style="text-align:left; padding:4px 8px;">Business context</th>
                        <th style="text-align:left; padding:4px 8px;">Notes</th>
                    </tr>
                </thead>
                <tbody>
                    {''.join(rows)}
                </tbody>
            </table>
        </details>
        """

    groups = group_rules_by_type(candidate_rules)

    state = {
        "group_index": 0,
        "history": [],
    }

    checkbox_state = {}

    title = widgets.HTML(
        value=f"<h4 style='margin:0;'>Human approval for AI-suggested DQ rules: {esc(table_name)}</h4>"
    )

    progress_label = widgets.HTML()
    group_summary = widgets.HTML()
    checkbox_box = widgets.VBox()
    business_context_box = widgets.HTML()
    rule_detail_box = widgets.HTML()

    severity_dropdown = widgets.Dropdown(
        options=["warning", "error"],
        description="Severity",
        layout=widgets.Layout(width="320px"),
    )

    group_note_box = widgets.Textarea(
        description="Group note",
        layout=widgets.Layout(width="780px", height="80px"),
    )

    apply_group_note_checkbox = widgets.Checkbox(
        value=False,
        description="Apply this group note to all selected rule descriptions",
        indent=False,
        layout=widgets.Layout(width="600px"),
    )

    params_box = widgets.Textarea(
        description="Rule config",
        layout=widgets.Layout(width="780px", height="120px"),
    )

    apply_button = widgets.Button(
        description="Apply selected columns",
        button_style="success",
        layout=widgets.Layout(width="260px"),
    )

    reject_group_button = widgets.Button(
        description="Reject this rule group",
        button_style="danger",
        layout=widgets.Layout(width="240px"),
    )

    undo_button = widgets.Button(
        description="Undo last group",
        button_style="warning",
        layout=widgets.Layout(width="200px"),
    )

    status = widgets.HTML()

    action_buttons = widgets.HBox([apply_button, reject_group_button, undo_button])

    form_box = widgets.VBox(
        [
            checkbox_box,
            business_context_box,
            rule_detail_box,
            severity_dropdown,
            group_note_box,
            apply_group_note_checkbox,
            params_box,
            action_buttons,
        ]
    )

    def current_group():
        idx = state["group_index"]
        if idx >= len(groups):
            return None
        return groups[idx]

    def update_progress():
        progress_label.value = (
            f"<b>Progress:</b> {state['group_index']} / {len(groups)} "
            f"&nbsp; | &nbsp; <b>Approved:</b> {len(APPROVED_RULES_FROM_WIDGET)} "
            f"&nbsp; | &nbsp; <b>Rejected:</b> {len(REJECTED_RULES_FROM_WIDGET)}"
        )

    def render_rule_details(rules):
        detail_rows = []

        for rule in rules:
            visible_params = collect_visible_rule_params(rule)
            params_text = json.dumps(visible_params, ensure_ascii=False) if visible_params else "{}"

            detail_rows.append(
                f"""
                <tr>
                    <td style="padding:4px 8px; vertical-align:top;"><code>{esc(rule.get("rule_id", ""))}</code></td>
                    <td style="padding:4px 8px; vertical-align:top;"><code>{esc(get_column_label(rule))}</code></td>
                    <td style="padding:4px 8px; vertical-align:top;">{esc(rule.get("severity", ""))}</td>
                    <td style="padding:4px 8px; vertical-align:top;">{esc(rule.get("description", ""))}</td>
                    <td style="padding:4px 8px; vertical-align:top;"><code>{esc(params_text)}</code></td>
                </tr>
                """
            )

        rule_detail_box.value = f"""
        <details style="margin-top:8px;">
            <summary style="cursor:pointer;">Show candidate rule details</summary>
            <table style="border-collapse:collapse; margin-top:8px; width:100%;">
                <thead>
                    <tr>
                        <th style="text-align:left; padding:4px 8px;">Rule ID</th>
                        <th style="text-align:left; padding:4px 8px;">Columns</th>
                        <th style="text-align:left; padding:4px 8px;">Severity</th>
                        <th style="text-align:left; padding:4px 8px;">Original description</th>
                        <th style="text-align:left; padding:4px 8px;">Editable config</th>
                    </tr>
                </thead>
                <tbody>
                    {''.join(detail_rows)}
                </tbody>
            </table>
        </details>
        """

    def load_current_group():
        update_progress()

        group = current_group()

        if group is None:
            total_reviewed = len(APPROVED_RULES_FROM_WIDGET) + len(REJECTED_RULES_FROM_WIDGET)

            group_summary.value = f"""
            <div style="
                border:1px solid #d1e7dd;
                background:#f0fff4;
                padding:14px;
                border-radius:8px;
                margin-top:8px;
            ">
                <div style="font-size:18px; font-weight:700; color:#166534; margin-bottom:6px;">
                    ✅ Review complete
                </div>
                <div><b>Table:</b> {esc(table_name)}</div>
                <div><b>Rule groups reviewed:</b> {len(groups)} / {len(groups)}</div>
                <div><b>Total candidate rules reviewed:</b> {total_reviewed} / {len(candidate_rules)}</div>
                <div><b>Approved:</b> {len(APPROVED_RULES_FROM_WIDGET)}</div>
                <div><b>Rejected:</b> {len(REJECTED_RULES_FROM_WIDGET)}</div>
                <div style="margin-top:10px; color:#444;">
                    Next step: store <b>APPROVED_RULES_FROM_WIDGET</b> into the metadata table.
                </div>
            </div>
            """

            form_box.layout.display = "none"
            status.value = (
                "<span style='color:#166534; font-weight:600;'>"
                "Approved rules are ready for metadata storage."
                "</span>"
            )
            return

        rule_type = group["rule_type"]
        rules = group["rules"]

        checkbox_state.clear()

        checkboxes = []
        for idx, rule in enumerate(rules):
            checkbox = widgets.Checkbox(
                value=True,
                description=get_column_label(rule),
                indent=False,
                layout=widgets.Layout(width="760px"),
            )
            checkbox_state[idx] = checkbox
            checkboxes.append(checkbox)

        checkbox_box.children = [
            widgets.HTML("<b>Apply this rule to these columns:</b>"),
            *checkboxes,
        ]

        first_rule = rules[0] if rules else {}

        severity_value = str(first_rule.get("severity", "warning")).lower()
        severity_dropdown.value = severity_value if severity_value in {"warning", "error"} else "warning"

        group_note_box.value = (
            f"Review note for {rule_type}. "
            "Tick the checkbox below only if this should replace every selected rule description."
        )

        apply_group_note_checkbox.value = False

        business_context_box.value = build_business_context_html(rules)
        render_rule_details(rules)

        params = build_common_params_from_rules(rules)

        if params:
            params_box.value = json.dumps(params, indent=2)
            params_box.layout.display = ""
        else:
            params_box.value = "{}"
            params_box.layout.display = "none"

        group_summary.value = f"""
        <div style="
            border:1px solid #e5e7eb;
            background:#fafafa;
            padding:12px;
            border-radius:8px;
            margin-top:8px;
        ">
            <div style="font-weight:700; margin-bottom:8px;">
                Rule group {state['group_index'] + 1} of {len(groups)}
            </div>
            <div><b>Rule group:</b> <code>{esc(rule_type)}</code></div>
            <div><b>Candidate columns:</b> {len(rules)}</div>
            <div style="margin-top:8px; color:#444;">
                Uncheck columns that should not use this rule, then batch apply.
            </div>
        </div>
        """

        form_box.layout.display = ""
        status.value = ""

    def build_rules_from_group(selected_only: bool):
        group = current_group()

        if group is None:
            status.value = "<span style='color:red'>No current rule group to review.</span>"
            return None, None

        try:
            raw_params = json.loads(params_box.value or "{}")
            if not isinstance(raw_params, dict):
                raise ValueError("Rule config must be a dictionary.")
        except Exception as exc:
            status.value = f"<span style='color:red'><b>Invalid Rule config JSON:</b> {esc(exc)}</span>"
            return None, None

        clean_params = clean_user_params(raw_params)

        approved = []
        rejected = []

        for idx, original_rule in enumerate(group["rules"]):
            checkbox = checkbox_state[idx]
            is_selected = bool(checkbox.value)

            edited_rule = copy.deepcopy(original_rule)
            edited_rule["severity"] = severity_dropdown.value

            if apply_group_note_checkbox.value:
                edited_rule["description"] = group_note_box.value
            else:
                edited_rule["description"] = original_rule.get("description", "")

            for key, value in clean_params.items():
                edited_rule[key] = value

            edited_rule = normalize_rule_params(edited_rule)

            if selected_only:
                if is_selected:
                    approved.append(edited_rule)
                else:
                    rejected.append(edited_rule)
            else:
                rejected.append(edited_rule)

        return approved, rejected

    def commit_group(approved, rejected):
        APPROVED_RULES_FROM_WIDGET.extend(approved)
        REJECTED_RULES_FROM_WIDGET.extend(rejected)

        state["history"].append(
            {
                "group_index": state["group_index"],
                "approved_count": len(approved),
                "rejected_count": len(rejected),
            }
        )

        state["group_index"] += 1
        load_current_group()

    def apply_clicked(_):
        approved, rejected = build_rules_from_group(selected_only=True)

        if approved is None:
            return

        if not approved and not rejected:
            status.value = "<span style='color:orange'>No rules found in this group.</span>"
            return

        commit_group(approved, rejected)

    def reject_group_clicked(_):
        approved, rejected = build_rules_from_group(selected_only=False)

        if approved is None:
            return

        commit_group([], rejected)

    def undo_clicked(_):
        if not state["history"]:
            status.value = "<span style='color:orange'>Nothing to undo.</span>"
            return

        last = state["history"].pop()

        if last["approved_count"]:
            del APPROVED_RULES_FROM_WIDGET[-last["approved_count"]:]

        if last["rejected_count"]:
            del REJECTED_RULES_FROM_WIDGET[-last["rejected_count"]:]

        state["group_index"] = last["group_index"]
        load_current_group()

    apply_button.on_click(apply_clicked)
    reject_group_button.on_click(reject_group_clicked)
    undo_button.on_click(undo_clicked)

    ui = widgets.VBox(
        [
            title,
            progress_label,
            group_summary,
            form_box,
            status,
        ],
        layout=widgets.Layout(
            border="1px solid #ddd",
            padding="12px",
            width="860px",
        ),
    )

    load_current_group()
    ipy_display(ui)

    return APPROVED_RULES_FROM_WIDGET, REJECTED_RULES_FROM_WIDGET

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
import json
import copy
import html
import ipywidgets as widgets
from IPython.display import display as ipy_display

RULE_DEACTIVATIONS_FROM_WIDGET = []
KEPT_ACTIVE_RULES_FROM_WIDGET = []


def launch_grouped_rule_deactivation_widget(active_rules: list[dict], table_name: str):
    """
    Review active DQ rules for possible deactivation.

    Workflow:
    1. One screen per rule type
    2. Active rules are shown as unchecked checkboxes
    3. User selects only rules/columns to deactivate
    4. Deactivation reason is required when deactivating selected rules
    5. User can keep all in the group and continue
    6. Search/filter is available across rule id, rule type, column, severity, and description

    Outputs:
    RULE_DEACTIVATIONS_FROM_WIDGET
    KEPT_ACTIVE_RULES_FROM_WIDGET

    Also returns:
    deactivations, kept_active_rules
    """

    global RULE_DEACTIVATIONS_FROM_WIDGET, KEPT_ACTIVE_RULES_FROM_WIDGET

    RULE_DEACTIVATIONS_FROM_WIDGET.clear()
    KEPT_ACTIVE_RULES_FROM_WIDGET.clear()

    def esc(value):
        return html.escape(str(value))

    def get_rule_type(rule):
        return str(rule.get("rule_type", "unknown"))

    def get_rule_columns(rule):
        cols = rule.get("columns", [])
        if isinstance(cols, list):
            return cols
        if cols is None:
            return []
        return [str(cols)]

    def get_column_label(rule):
        cols = get_rule_columns(rule)
        return ", ".join(str(c) for c in cols) if cols else "(no column)"

    def searchable_text(rule):
        parts = [
            rule.get("rule_id", ""),
            rule.get("rule_type", ""),
            ",".join(get_rule_columns(rule)),
            rule.get("severity", ""),
            rule.get("description", ""),
        ]

        for key, value in rule.items():
            if key not in {"rule_id", "rule_type", "columns", "severity", "description"}:
                parts.append(f"{key}={value}")

        return " ".join(str(p) for p in parts).lower()

    def group_rules_by_type(rules):
        grouped = {}
        for rule in rules:
            rule_type = get_rule_type(rule)
            grouped.setdefault(rule_type, []).append(rule)

        return [
            {"rule_type": rule_type, "rules": rules_for_type}
            for rule_type, rules_for_type in grouped.items()
        ]

    def rule_params(rule):
        skip_keys = {"rule_id", "rule_type", "columns", "severity", "description"}
        return {k: v for k, v in rule.items() if k not in skip_keys}

    groups = group_rules_by_type(active_rules)

    state = {
        "group_index": 0,
        "history": [],
        "search": "",
    }

    checkbox_state = {}

    title = widgets.HTML(
        value=f"<h4 style='margin:0;'>DQ rule administration: {esc(table_name)}</h4>"
    )

    progress_label = widgets.HTML()
    group_summary = widgets.HTML()

    search_box = widgets.Text(
        value="",
        placeholder="Search rule id, rule type, column, severity, description...",
        description="Search",
        layout=widgets.Layout(width="780px"),
    )

    checkbox_box = widgets.VBox()
    rule_detail_box = widgets.HTML()

    reason_box = widgets.Textarea(
        description="Reason",
        placeholder="Required when deactivating selected rules.",
        layout=widgets.Layout(width="780px", height="90px"),
    )

    deactivate_selected_button = widgets.Button(
        description="Deactivate selected rules",
        button_style="danger",
        layout=widgets.Layout(width="260px"),
    )

    keep_group_button = widgets.Button(
        description="Keep all in this group",
        button_style="success",
        layout=widgets.Layout(width="240px"),
    )

    undo_button = widgets.Button(
        description="Undo last group",
        button_style="warning",
        layout=widgets.Layout(width="200px"),
    )

    status = widgets.HTML()

    action_buttons = widgets.HBox(
        [deactivate_selected_button, keep_group_button, undo_button]
    )

    form_box = widgets.VBox(
        [
            search_box,
            checkbox_box,
            rule_detail_box,
            reason_box,
            action_buttons,
        ]
    )

    def current_group():
        idx = state["group_index"]
        if idx >= len(groups):
            return None
        return groups[idx]

    def filtered_rules_for_group(group):
        search = state["search"].strip().lower()
        rules = group["rules"]

        if not search:
            return list(enumerate(rules))

        return [
            (idx, rule)
            for idx, rule in enumerate(rules)
            if search in searchable_text(rule)
        ]

    def update_progress():
        progress_label.value = (
            f"<b>Progress:</b> {state['group_index']} / {len(groups)} "
            f"&nbsp; | &nbsp; <b>Kept active:</b> {len(KEPT_ACTIVE_RULES_FROM_WIDGET)} "
            f"&nbsp; | &nbsp; <b>Marked for deactivation:</b> {len(RULE_DEACTIVATIONS_FROM_WIDGET)}"
        )

    def render_group_details(group, visible_rule_pairs):
        detail_rows = []

        for _, rule in visible_rule_pairs:
            params = rule_params(rule)
            params_text = json.dumps(params, ensure_ascii=False) if params else "{}"

            detail_rows.append(
                f"""
                <tr>
                    <td style="padding:4px 8px;"><code>{esc(rule.get("rule_id", ""))}</code></td>
                    <td style="padding:4px 8px;"><code>{esc(get_column_label(rule))}</code></td>
                    <td style="padding:4px 8px;">{esc(rule.get("severity", ""))}</td>
                    <td style="padding:4px 8px;">{esc(rule.get("description", ""))}</td>
                    <td style="padding:4px 8px;"><code>{esc(params_text)}</code></td>
                </tr>
                """
            )

        if not detail_rows:
            detail_rows.append(
                """
                <tr>
                    <td colspan="5" style="padding:8px; color:#777;">
                        No rules match the current search.
                    </td>
                </tr>
                """
            )

        rule_detail_box.value = f"""
        <details style="margin-top:8px;">
            <summary style="cursor:pointer;">Show active rule details</summary>
            <table style="border-collapse:collapse; margin-top:8px; width:100%;">
                <thead>
                    <tr>
                        <th style="text-align:left; padding:4px 8px;">Rule ID</th>
                        <th style="text-align:left; padding:4px 8px;">Columns</th>
                        <th style="text-align:left; padding:4px 8px;">Severity</th>
                        <th style="text-align:left; padding:4px 8px;">Description</th>
                        <th style="text-align:left; padding:4px 8px;">Parameters</th>
                    </tr>
                </thead>
                <tbody>
                    {''.join(detail_rows)}
                </tbody>
            </table>
        </details>
        """

    def render_checkboxes(group):
        visible_rule_pairs = filtered_rules_for_group(group)

        checkbox_state.clear()
        checkboxes = []

        for original_idx, rule in visible_rule_pairs:
            checkbox = widgets.Checkbox(
                value=False,
                description=f"{get_column_label(rule)}  |  {rule.get('rule_id', '')}",
                indent=False,
                layout=widgets.Layout(width="760px"),
            )

            checkbox_state[original_idx] = checkbox
            checkboxes.append(checkbox)

        if checkboxes:
            checkbox_box.children = [
                widgets.HTML(
                    "<b>Select active rules to deactivate:</b> "
                    "<span style='color:#555;'>(default is keep active)</span>"
                ),
                *checkboxes,
            ]
        else:
            checkbox_box.children = [
                widgets.HTML(
                    "<b>Select active rules to deactivate:</b><br>"
                    "<span style='color:#777;'>No active rules match the current search.</span>"
                )
            ]

        render_group_details(group, visible_rule_pairs)

    def load_current_group():
        update_progress()

        group = current_group()

        if group is None:
            total_reviewed = (
                len(KEPT_ACTIVE_RULES_FROM_WIDGET)
                + len(RULE_DEACTIVATIONS_FROM_WIDGET)
            )

            group_summary.value = f"""
            <div style="
                border:1px solid #dbeafe;
                background:#eff6ff;
                padding:14px;
                border-radius:8px;
                margin-top:8px;
            ">
                <div style="font-size:18px; font-weight:700; color:#1d4ed8; margin-bottom:6px;">
                    ✅ Administration review complete
                </div>
                <div><b>Table:</b> {esc(table_name)}</div>
                <div><b>Rule groups reviewed:</b> {len(groups)} / {len(groups)}</div>
                <div><b>Total active rules reviewed:</b> {total_reviewed} / {len(active_rules)}</div>
                <div><b>Kept active:</b> {len(KEPT_ACTIVE_RULES_FROM_WIDGET)}</div>
                <div><b>Marked for deactivation:</b> {len(RULE_DEACTIVATIONS_FROM_WIDGET)}</div>
                <div style="margin-top:10px; color:#444;">
                    Next step: append <b>RULE_DEACTIVATIONS_FROM_WIDGET</b> into the metadata table as inactive versions.
                </div>
            </div>
            """

            form_box.layout.display = "none"

            if total_reviewed != len(active_rules):
                status.value = (
                    f"<span style='color:#b91c1c; font-weight:600;'>"
                    f"Warning: reviewed {total_reviewed} of {len(active_rules)} active rules."
                    f"</span>"
                )
            else:
                status.value = (
                    "<span style='color:#1d4ed8; font-weight:600;'>"
                    "Deactivation decisions are ready for metadata append."
                    "</span>"
                )

            return

        rule_type = group["rule_type"]
        rules = group["rules"]

        search_box.value = ""
        state["search"] = ""
        reason_box.value = ""

        group_summary.value = f"""
        <div style="
            border:1px solid #e5e7eb;
            background:#fafafa;
            padding:12px;
            border-radius:8px;
            margin-top:8px;
        ">
            <div style="font-weight:700; margin-bottom:8px;">
                Rule group {state['group_index'] + 1} of {len(groups)}
            </div>
            <div><b>Rule group:</b> <code>{esc(rule_type)}</code></div>
            <div><b>Active rules in group:</b> {len(rules)}</div>
            <div style="margin-top:8px; color:#444;">
                Select only the rules/columns you want to deactivate. Leave unchecked to keep active.
            </div>
        </div>
        """

        render_checkboxes(group)

        form_box.layout.display = ""
        status.value = ""

    def build_decisions_for_group(deactivate_selected: bool):
        group = current_group()

        if group is None:
            status.value = "<span style='color:red'>No current rule group to review.</span>"
            return None, None

        deactivations = []
        kept_rules = []

        selected_indices = {
            idx for idx, checkbox in checkbox_state.items() if checkbox.value
        }

        if deactivate_selected and not selected_indices:
            status.value = (
                "<span style='color:orange'>Select at least one rule to deactivate, "
                "or choose keep all in this group.</span>"
            )
            return None, None

        reason = reason_box.value.strip()

        if deactivate_selected and not reason:
            status.value = (
                "<span style='color:red'><b>Deactivation reason is required.</b></span>"
            )
            return None, None

        for idx, rule in enumerate(group["rules"]):
            rule_copy = copy.deepcopy(rule)

            if deactivate_selected and idx in selected_indices:
                deactivations.append(
                    {
                        "rule": rule_copy,
                        "action_reason": reason,
                    }
                )
            else:
                kept_rules.append(rule_copy)

        return deactivations, kept_rules

    def commit_group(deactivations, kept_rules):
        RULE_DEACTIVATIONS_FROM_WIDGET.extend(deactivations)
        KEPT_ACTIVE_RULES_FROM_WIDGET.extend(kept_rules)

        state["history"].append(
            {
                "group_index": state["group_index"],
                "deactivation_count": len(deactivations),
                "kept_count": len(kept_rules),
            }
        )

        state["group_index"] += 1
        load_current_group()

    def deactivate_selected_clicked(_):
        deactivations, kept_rules = build_decisions_for_group(deactivate_selected=True)

        if deactivations is None:
            return

        commit_group(deactivations, kept_rules)

    def keep_group_clicked(_):
        deactivations, kept_rules = build_decisions_for_group(deactivate_selected=False)

        if deactivations is None:
            return

        commit_group([], kept_rules)

    def undo_clicked(_):
        if not state["history"]:
            status.value = "<span style='color:orange'>Nothing to undo.</span>"
            return

        last = state["history"].pop()

        if last["deactivation_count"]:
            del RULE_DEACTIVATIONS_FROM_WIDGET[-last["deactivation_count"]:]

        if last["kept_count"]:
            del KEPT_ACTIVE_RULES_FROM_WIDGET[-last["kept_count"]:]

        state["group_index"] = last["group_index"]
        load_current_group()

    def search_changed(change):
        group = current_group()

        if group is None:
            return

        state["search"] = change["new"]
        render_checkboxes(group)

    search_box.observe(search_changed, names="value")

    def reason_changed(change):
        if str(change["new"]).strip():
            status.value = ""

    reason_box.observe(reason_changed, names="value")

    deactivate_selected_button.on_click(deactivate_selected_clicked)
    keep_group_button.on_click(keep_group_clicked)
    undo_button.on_click(undo_clicked)

    ui = widgets.VBox(
        [
            title,
            progress_label,
            group_summary,
            form_box,
            status,
        ],
        layout=widgets.Layout(
            border="1px solid #ddd",
            padding="12px",
            width="850px",
        ),
    )

    load_current_group()
    ipy_display(ui)

    return RULE_DEACTIVATIONS_FROM_WIDGET, KEPT_ACTIVE_RULES_FROM_WIDGET


StatementMeta(, , -1, Waiting, , Waiting, True)

## 4. Data quality enforcement helpers


In [ ]:
def _validate_common_rule_fields(
    rules: list[dict[str, Any]],
    allowed_types: set[str],
    label: str,
) -> list[dict[str, Any]]:
    if not isinstance(rules, list):
        raise ValueError(f"{label} must be a list of dictionaries.")

    required_fields = {"rule_id", "rule_type", "severity", "description"}

    for index, rule in enumerate(rules):
        if not isinstance(rule, dict):
            raise ValueError(f"{label} entry at index {index} must be a dictionary.")

        missing = required_fields.difference(rule)
        if missing:
            raise ValueError(f"{label} '{rule.get('rule_id', index)}' is missing fields: {sorted(missing)}")

        rule_id = rule["rule_id"]
        rule_type = rule["rule_type"]
        severity = str(rule["severity"]).lower()

        if rule_type not in allowed_types:
            raise ValueError(f"{label} '{rule_id}' has unsupported rule_type '{rule_type}'.")

        if severity not in {"warning", "error"}:
            raise ValueError(f"{label} '{rule_id}' severity must be warning or error.")

    return rules


def validate_dq_rules(rules: list[dict[str, Any]]) -> list[dict[str, Any]]:
    _validate_common_rule_fields(rules, AI_SUGGESTABLE_DQ_RULE_TYPES, "DQ rule")

    for rule in rules:
        rule_id = rule["rule_id"]
        rule_type = rule["rule_type"]
        columns = rule.get("columns", [])

        if not isinstance(columns, list) or len(columns) != 1:
            raise ValueError(f"DQ rule '{rule_id}' must have exactly one column in `columns`.")

        if rule_type == "accepted_values" and "allowed_values" not in rule:
            raise ValueError(f"DQ rule '{rule_id}' requires allowed_values.")

        if rule_type == "value_range" and "lower_bound" not in rule and "upper_bound" not in rule:
            raise ValueError(f"DQ rule '{rule_id}' requires lower_bound or upper_bound.")

        if rule_type == "regex_format" and "regex_pattern" not in rule:
            raise ValueError(f"DQ rule '{rule_id}' requires regex_pattern.")

    return rules


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
DQ_RESULT_SCHEMA = StructType(
    [
        StructField("table_name", StringType(), True),
        StructField("rule_id", StringType(), True),
        StructField("rule_type", StringType(), True),
        StructField("columns", StringType(), True),
        StructField("severity", StringType(), True),
        StructField("status", StringType(), True),
        StructField("failed_count", LongType(), True),
        StructField("total_count", LongType(), True),
        StructField("failed_percent", DoubleType(), True),
        StructField("description", StringType(), True),
        StructField("run_timestamp", StringType(), True),
        StructField("details", StringType(), True),
    ]
)


def _quality_result_row(
    table_name: str,
    rule: dict[str, Any],
    failed_count: int,
    total_count: int,
    details: str = "",
):
    return {
        "table_name": table_name,
        "rule_id": str(rule["rule_id"]),
        "rule_type": str(rule["rule_type"]),
        "columns": ",".join(rule.get("columns", [])),
        "severity": str(rule.get("severity", "warning")),
        "status": "PASS" if failed_count == 0 else "FAIL",
        "failed_count": int(failed_count),
        "total_count": int(total_count),
        "failed_percent": float(round((failed_count / total_count) * 100, 4)) if total_count else 0.0,
        "description": str(rule.get("description", "")),
        "run_timestamp": datetime.now(timezone.utc).isoformat(),
        "details": details,
    }


def run_dq_rules(
    df,
    table_name: str,
    rules: list[dict[str, Any]],
):
    """
    Run approved DQ rules loaded from the metadata table.
    """
    validate_dq_rules(rules)

    spark = df.sparkSession
    total_count = df.count()
    rows = []

    for rule in rules:
        rule_type = rule["rule_type"]
        col_name = rule["columns"][0]

        if rule_type == "not_null":
            failed_count = df.filter(F.col(col_name).isNull()).count()
            rows.append(_quality_result_row(table_name, rule, failed_count, total_count))

        elif rule_type == "unique_key":
            failed_count = (
                df.groupBy(col_name)
                .count()
                .filter(F.col("count") > 1)
                .agg(F.sum("count").alias("failed_count"))
                .collect()[0]["failed_count"]
            )
            rows.append(_quality_result_row(table_name, rule, int(failed_count or 0), total_count))

        elif rule_type == "accepted_values":
            failed_count = (
                df.filter(F.col(col_name).isNotNull() & ~F.col(col_name).isin(rule["allowed_values"]))
                .count()
            )
            rows.append(_quality_result_row(table_name, rule, failed_count, total_count))

        elif rule_type == "value_range":
            failed_condition = F.lit(False)

            if rule.get("lower_bound") is not None:
                failed_condition = failed_condition | (
                    F.col(col_name).cast("double") < F.lit(float(rule["lower_bound"]))
                )

            if rule.get("upper_bound") is not None:
                failed_condition = failed_condition | (
                    F.col(col_name).cast("double") > F.lit(float(rule["upper_bound"]))
                )

            failed_count = df.filter(
                F.col(col_name).isNotNull() & failed_condition
            ).count()

            rows.append(_quality_result_row(table_name, rule, failed_count, total_count))

        elif rule_type == "regex_format":
            failed_count = (
                df.filter(F.col(col_name).isNotNull() & ~F.col(col_name).rlike(rule["regex_pattern"]))
                .count()
            )
            rows.append(_quality_result_row(table_name, rule, failed_count, total_count))

    return spark.createDataFrame(rows, schema=DQ_RESULT_SCHEMA)


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from datetime import datetime, timezone
import uuid
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def split_valid_quarantine_and_failures(
    df,
    rules: list[dict[str, Any]],
    dq_run_id: str | None = None,
    row_id_columns: list[str] | None = None,
):
    """
    Split a DataFrame into accepted rows, quarantined rows, and row-rule failure evidence.

    Returns
    -------
    tuple
        (df_valid, df_quarantine_rows, df_quarantine_failures)

    Design
    ------
    df_valid:
        Rows that pass all row-level DQ rules.

    df_quarantine_rows:
        Original rows that fail at least one row-level DQ rule.
        One row appears once, even if it fails multiple rules.

    df_quarantine_failures:
        One row per failed rule per quarantined source row.
        This is the proper evidence table for remediation and audit.

    Notes
    -----
    - dq_run_id ties all records to one DQ run.
    - dq_row_id ties the accepted/quarantined/failure records back to the source row for that run.
    - dq_quarantine_id ties one quarantined source row to all of its failed rule records.
    """
    validate_dq_rules(rules)

    if dq_run_id is None:
        dq_run_id = str(uuid.uuid4())

    run_ts = datetime.now(timezone.utc).isoformat()

    # Add a stable row id for this run.
    # If business keys are supplied, hash them.
    # Otherwise, use monotonically_increasing_id for demo/runtime traceability.
    if row_id_columns:
        df_with_ids = df.withColumn(
            "dq_row_id",
            F.sha2(
                F.concat_ws(
                    "||",
                    *[
                        F.coalesce(F.col(c).cast("string"), F.lit("<NULL>"))
                        for c in row_id_columns
                    ],
                ),
                256,
            ),
        )
    else:
        df_with_ids = df.withColumn(
            "dq_row_id",
            F.sha2(
                F.concat_ws(
                    "||",
                    *[
                        F.coalesce(F.col(c).cast("string"), F.lit("<NULL>"))
                        for c in df.columns
                    ],
                    F.monotonically_increasing_id().cast("string"),
                ),
                256,
            ),
        )

    df_with_ids = df_with_ids.withColumn("dq_run_id", F.lit(dq_run_id))

    failure_dfs = []

    working = df_with_ids

    for rule in rules:
        rule_id = str(rule["rule_id"])
        rule_type = str(rule["rule_type"])
        columns = rule["columns"]
        col_name = columns[0]
        severity = str(rule.get("severity", "warning"))
        description = str(rule.get("description", ""))

        if rule_type == "not_null":
            failed = F.col(col_name).isNull() | (F.trim(F.col(col_name).cast("string")) == "")

        elif rule_type == "unique_key":
            duplicate_count_col = f"__dq_duplicate_count_{rule_id}"
            working = working.withColumn(
                duplicate_count_col,
                F.count(F.lit(1)).over(Window.partitionBy(*[F.col(c) for c in columns])),
            )
            failed = F.col(duplicate_count_col) > F.lit(1)

        elif rule_type == "accepted_values":
            failed = F.col(col_name).isNotNull() & ~F.col(col_name).isin(rule["allowed_values"])

        elif rule_type == "value_range":
            failed_condition = F.lit(False)

            if rule.get("min_value") is not None:
                failed_condition = failed_condition | (
                    F.col(col_name).cast("double") < F.lit(float(rule["min_value"]))
                )

            if rule.get("max_value") is not None:
                failed_condition = failed_condition | (
                    F.col(col_name).cast("double") > F.lit(float(rule["max_value"]))
                )

            failed = F.col(col_name).isNotNull() & failed_condition

        elif rule_type == "regex_format":
            failed = F.col(col_name).isNotNull() & ~F.col(col_name).rlike(rule["regex_pattern"])

        else:
            # Contract guardrails are not row-level quarantine rules.
            continue

        failed = F.coalesce(failed, F.lit(False))

        failure_df = (
            working
            .filter(failed)
            .select(
                F.col("dq_run_id"),
                F.col("dq_row_id"),
                F.lit(rule_id).alias("rule_id"),
                F.lit(rule_type).alias("rule_type"),
                F.lit(",".join(columns)).alias("failed_columns"),
                F.lit(severity).alias("severity"),
                F.lit(description).alias("description"),
                F.lit(run_ts).alias("dq_failed_ts"),
            )
        )

        failure_dfs.append(failure_df)

        if rule_type == "unique_key":
            working = working.drop(duplicate_count_col)

    if not failure_dfs:
        empty_failures = df.sparkSession.createDataFrame(
            [],
            """
            dq_run_id string,
            dq_row_id string,
            dq_quarantine_id string,
            rule_id string,
            rule_type string,
            failed_columns string,
            severity string,
            description string,
            dq_failed_ts string
            """,
        )
        return df_with_ids, df.limit(0), empty_failures

    df_quarantine_failures = failure_dfs[0]
    for item in failure_dfs[1:]:
        df_quarantine_failures = df_quarantine_failures.unionByName(item)

    # One quarantine id per bad source row in this DQ run.
    quarantine_ids = (
        df_quarantine_failures
        .select("dq_run_id", "dq_row_id")
        .distinct()
        .withColumn(
            "dq_quarantine_id",
            F.sha2(F.concat_ws("||", F.col("dq_run_id"), F.col("dq_row_id")), 256),
        )
    )

    df_quarantine_failures = (
        df_quarantine_failures
        .join(quarantine_ids, on=["dq_run_id", "dq_row_id"], how="left")
        .select(
            "dq_run_id",
            "dq_row_id",
            "dq_quarantine_id",
            "rule_id",
            "rule_type",
            "failed_columns",
            "severity",
            "description",
            "dq_failed_ts",
        )
    )

    df_quarantine_rows = (
        working
        .join(quarantine_ids, on=["dq_run_id", "dq_row_id"], how="inner")
        .withColumn("dq_quarantine_ts", F.lit(run_ts))
    )

    df_valid = (
        working
        .join(quarantine_ids.select("dq_run_id", "dq_row_id"), on=["dq_run_id", "dq_row_id"], how="left_anti")
    )

    return df_valid, df_quarantine_rows, df_quarantine_failures


StatementMeta(, , -1, Waiting, , Waiting, True)

## 5. Governance functions

Governance classification uses approved business context as read-only evidence.


In [ ]:
DEFAULT_GOVERNANCE_WIDGET_CONFIG = {
    "confidentiality_labels": ["public", "confidential", "restricted"],
    "personal_identifier_options": ["not_personal_data", "direct_identifier", "indirect_identifier", "unknown"],
}

COLUMN_GOVERNANCE_CONTEXT_FROM_WIDGET = []

PDPA_PERSONAL_IDENTIFIER_PROMPT = """
You are helping classify whether a data column may identify an individual under PDPA-style personal data rules.

Definition:
Personal data is any data that can directly or indirectly identify an individual, whether true or not.

Classification options:
- not_personal_data: The column does not identify an individual by itself or in normal combination.
- direct_identifier: The column explicitly identifies a person, such as full name, email, NRIC, passport number, student/staff ID, phone number, biometric data.
- indirect_identifier: The column may identify a person when combined with other fields, such as precise location, age, occupation, small group membership, department, timestamp, or rare attributes.
- unknown: There is not enough context to decide.

Use the approved column business context heavily. The technical profile alone is not enough.

Return ONLY a valid Python dictionary with this exact shape:
{
  "column_name": "<column_name>",
  "personal_identifier_classification": "not_personal_data|direct_identifier|indirect_identifier|unknown",
  "reason": "<short reason>"
}

Column profile:
Table name: {table_name}
Column name: {column_name}
Data type: {data_type}
Row count: {row_count}
Null count: {null_count}
Distinct count: {distinct_count}
Min value: {min_value}
Max value: {max_value}
Observed values sample: {observed_values_sample}

Approved business context:
{business_context}

Business context notes:
{business_context_notes}
"""


def prepare_governance_profile_input(profile_df, table_name: str, column_contexts=None):
    """Join approved business context into profile rows before governance AI classification."""
    cols = set(profile_df.columns)
    if {"column_name", "data_type"}.issubset(cols):
        out = profile_df
    elif {"COLUMN_NAME", "DATA_TYPE"}.issubset(cols):
        out = profile_df.select(
            F.lit(table_name).alias("table_name"),
            F.col("COLUMN_NAME").alias("column_name"),
            F.col("DATA_TYPE").alias("data_type"),
            F.col("ROW_COUNT").alias("row_count") if "ROW_COUNT" in cols else F.lit(None).alias("row_count"),
            F.col("NULL_COUNT").alias("null_count") if "NULL_COUNT" in cols else F.lit(None).alias("null_count"),
            F.col("DISTINCT_COUNT").alias("distinct_count") if "DISTINCT_COUNT" in cols else F.lit(None).alias("distinct_count"),
            F.col("MIN_VALUE").alias("min_value") if "MIN_VALUE" in cols else F.lit(None).alias("min_value"),
            F.col("MAX_VALUE").alias("max_value") if "MAX_VALUE" in cols else F.lit(None).alias("max_value"),
        )
    else:
        raise ValueError("profile_df must contain either column_name/data_type or COLUMN_NAME/DATA_TYPE.")

    required_defaults = {
        "table_name": F.lit(table_name),
        "row_count": F.lit(None),
        "null_count": F.lit(None),
        "distinct_count": F.lit(None),
        "min_value": F.lit(None),
        "max_value": F.lit(None),
        "observed_values_sample": F.lit(""),
    }
    out_cols = set(out.columns)
    for col_name, default_expr in required_defaults.items():
        if col_name not in out_cols:
            out = out.withColumn(col_name, default_expr)

    out = out.select("table_name", "column_name", "data_type", "row_count", "null_count", "distinct_count", "min_value", "max_value", "observed_values_sample")
    profile_columns = [r["column_name"] for r in out.select("column_name").collect()]
    context_df = column_context_rows_for_spark(profile_df.sparkSession, profile_columns, column_contexts)
    return (
        out.join(context_df, on="column_name", how="left")
        .select(
            "table_name",
            "column_name",
            "data_type",
            "row_count",
            "null_count",
            "distinct_count",
            "min_value",
            "max_value",
            "observed_values_sample",
            F.coalesce(F.col("business_context"), F.lit("")).alias("business_context"),
            F.coalesce(F.col("business_context_notes"), F.lit("")).alias("business_context_notes"),
        )
    )


def suggest_personal_identifier_classifications(
    profile_df,
    table_name: str,
    column_contexts=None,
    prompt_template: str = PDPA_PERSONAL_IDENTIFIER_PROMPT,
    output_col: str = "ai_response",
):
    """Use Fabric AI Functions to suggest PDPA-style personal identifier classification per column."""
    prepared = prepare_governance_profile_input(profile_df, table_name, column_contexts)
    return prepared.ai.generate_response(prompt=prompt_template, output_col=output_col)


def extract_personal_identifier_suggestions(response_df, response_col: str = "ai_response") -> list[dict]:
    """Convert Fabric AI response rows into list[dict] for the governance review widget."""
    allowed = {"not_personal_data", "direct_identifier", "indirect_identifier", "unknown"}
    suggestions = []
    for row in response_df.select(response_col).collect():
        parsed = _parse_ai_dict_response(row[response_col])
        if not parsed:
            continue
        column_name = parsed.get("column_name") or parsed.get("COLUMN_NAME")
        classification = parsed.get("personal_identifier_classification") or parsed.get("classification")
        reason = parsed.get("reason") or parsed.get("explanation") or ""
        if not column_name:
            continue
        if classification not in allowed:
            classification = "unknown"
        suggestions.append({"column_name": str(column_name), "personal_identifier_classification": classification, "reason": str(reason)})
    return suggestions


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
def _get_governance_ai_suggestion(ai_suggestion_map, column_name):
    suggestion = ai_suggestion_map.get(column_name, {})
    if not isinstance(suggestion, dict):
        return {}
    suggested_class = (
        suggestion.get("personal_identifier_classification")
        or suggestion.get("ai_suggested_personal_identifier_classification")
        or suggestion.get("classification")
    )
    reason = suggestion.get("reason") or suggestion.get("ai_suggested_reason") or suggestion.get("explanation") or ""
    return {"personal_identifier_classification": suggested_class, "reason": reason}


def review_column_governance_context(
    profile_df=None,
    columns=None,
    table_name: str | None = None,
    environment_name: str | None = None,
    dataset_name: str | None = None,
    column_contexts=None,
    governance_config=None,
    ai_suggestions=None,
):
    """Review governance classifications. Business context is read-only evidence."""
    global COLUMN_GOVERNANCE_CONTEXT_FROM_WIDGET
    COLUMN_GOVERNANCE_CONTEXT_FROM_WIDGET.clear()

    governance_config = governance_config or DEFAULT_GOVERNANCE_WIDGET_CONFIG
    confidentiality_options = governance_config.get("confidentiality_labels", DEFAULT_GOVERNANCE_WIDGET_CONFIG["confidentiality_labels"])
    personal_identifier_options = governance_config.get("personal_identifier_options", DEFAULT_GOVERNANCE_WIDGET_CONFIG["personal_identifier_options"])
    if not confidentiality_options:
        raise ValueError("governance_config['confidentiality_labels'] cannot be empty.")
    if not personal_identifier_options:
        raise ValueError("governance_config['personal_identifier_options'] cannot be empty.")

    if columns is None:
        if profile_df is None:
            raise ValueError("Provide either profile_df or columns.")
        columns = _extract_columns_from_profile(profile_df)
    columns = [str(c) for c in columns]

    context_map = normalise_records_by_column(column_contexts)
    ai_suggestion_map = normalise_records_by_column(ai_suggestions)
    state = {"index": 0, "history": []}

    title = widgets.HTML(value=f"<h4 style='margin:0;'>Column governance classification review: {_esc(table_name or '')}</h4>")
    progress_label = widgets.HTML()
    column_summary = widgets.HTML()
    column_name_box = widgets.Text(description="Column", disabled=True, layout=widgets.Layout(width="650px"))
    business_context_view = widgets.HTML()
    confidentiality_dropdown = widgets.Dropdown(options=confidentiality_options, value=confidentiality_options[0], description="Confidentiality", layout=widgets.Layout(width="460px"))
    ai_suggestion_box = widgets.HTML()
    personal_identifier_dropdown = widgets.Dropdown(
        options=personal_identifier_options,
        value="unknown" if "unknown" in personal_identifier_options else personal_identifier_options[0],
        description="Personal ID",
        layout=widgets.Layout(width="520px"),
    )
    notes_box = widgets.Textarea(description="Governance notes", placeholder="Optional governance notes, assumptions, or review comments.", layout=widgets.Layout(width="780px", height="90px"))
    save_button = widgets.Button(description="Approve / Save Governance", button_style="success", layout=widgets.Layout(width="260px"))
    skip_button = widgets.Button(description="Skip Column", button_style="warning", layout=widgets.Layout(width="180px"))
    undo_button = widgets.Button(description="Undo Last", layout=widgets.Layout(width="160px"))
    status = widgets.HTML()
    form_box = widgets.VBox([column_name_box, business_context_view, confidentiality_dropdown, ai_suggestion_box, personal_identifier_dropdown, notes_box, widgets.HBox([save_button, skip_button, undo_button])])

    def current_column():
        idx = state["index"]
        return None if idx >= len(columns) else columns[idx]

    def load_current_column():
        col = current_column()
        progress_label.value = f"<b>Progress:</b> {state['index']} / {len(columns)} &nbsp; | &nbsp; <b>Saved:</b> {len(COLUMN_GOVERNANCE_CONTEXT_FROM_WIDGET)}"
        if col is None:
            column_summary.value = f"""
            <div style="border:1px solid #d1e7dd;background:#f0fff4;padding:14px;border-radius:8px;margin-top:8px;">
                <div style="font-size:18px;font-weight:700;color:#166534;margin-bottom:6px;">✅ Column governance review complete</div>
                <div><b>Table:</b> {_esc(table_name or '')}</div>
                <div><b>Columns reviewed:</b> {state['index']} / {len(columns)}</div>
                <div><b>Saved governance rows:</b> {len(COLUMN_GOVERNANCE_CONTEXT_FROM_WIDGET)}</div>
                <div style="margin-top:10px;color:#444;">Next step: write <b>COLUMN_GOVERNANCE_CONTEXT_FROM_WIDGET</b> to METADATA_COLUMN_GOVERNANCE.</div>
            </div>"""
            form_box.layout.display = "none"
            status.value = "<span style='color:#166534;font-weight:600;'>Column governance classifications are ready for metadata write.</span>"
            return

        context = context_map.get(col, {}) if isinstance(context_map.get(col, {}), dict) else {}
        business_context = context.get("business_context", "")
        business_context_notes = context.get("notes", "") or context.get("business_context_notes", "")
        suggestion = _get_governance_ai_suggestion(ai_suggestion_map, col)
        suggested_class = suggestion.get("personal_identifier_classification")
        suggested_reason = suggestion.get("reason") or ""

        column_summary.value = f"""
        <div style="border:1px solid #e5e7eb;background:#fafafa;padding:12px;border-radius:8px;margin-top:8px;">
            <div style="font-weight:700;margin-bottom:8px;">Column {state['index'] + 1} of {len(columns)}</div>
            <div><b>Column:</b> <code>{_esc(col)}</code></div>
            <div style="margin-top:8px;color:#444;">Approve governance classification using read-only business context and AI suggestion.</div>
        </div>"""
        column_name_box.value = col

        if business_context or business_context_notes:
            business_context_view.value = f"""
            <div style="border:1px solid #e5e7eb;background:#fafafa;padding:10px;border-radius:8px;margin-top:6px;margin-bottom:6px;">
                <div style="font-weight:700;">Approved business context</div>
                <div><b>Context:</b> {_esc(business_context)}</div>
                <div><b>Notes:</b> {_esc(business_context_notes)}</div>
                <div style="margin-top:6px;color:#666;">Read-only. To change this, rerun review_business_context(...).</div>
            </div>"""
        else:
            business_context_view.value = """
            <div style="border:1px solid #fde68a;background:#fffbeb;padding:10px;border-radius:8px;margin-top:6px;margin-bottom:6px;">
                <div style="font-weight:700;color:#92400e;">No approved business context found</div>
                <div style="color:#666;">Consider running review_business_context(...) before governance review.</div>
            </div>"""

        confidentiality_dropdown.value = confidentiality_options[0]
        if suggested_class:
            ai_suggestion_box.value = f"""
            <div style="border:1px solid #dbeafe;background:#eff6ff;padding:10px;border-radius:8px;margin-top:6px;margin-bottom:6px;">
                <div style="font-weight:700;color:#1d4ed8;">AI suggested personal identifier classification</div>
                <div><b>Suggested:</b> <code>{_esc(suggested_class)}</code></div>
                <div><b>Reason:</b> {_esc(suggested_reason)}</div>
                <div style="margin-top:6px;color:#444;">Advisory only. The dropdown value below is the approved value.</div>
            </div>"""
            if suggested_class in personal_identifier_options:
                personal_identifier_dropdown.value = suggested_class
            elif "unknown" in personal_identifier_options:
                personal_identifier_dropdown.value = "unknown"
            else:
                personal_identifier_dropdown.value = personal_identifier_options[0]
        else:
            ai_suggestion_box.value = """
            <div style="border:1px solid #e5e7eb;background:#fafafa;padding:10px;border-radius:8px;margin-top:6px;margin-bottom:6px;">
                <div style="font-weight:700;">AI suggested personal identifier classification</div>
                <div style="color:#666;">No AI suggestion supplied for this column.</div>
            </div>"""
            personal_identifier_dropdown.value = "unknown" if "unknown" in personal_identifier_options else personal_identifier_options[0]
        notes_box.value = ""
        form_box.layout.display = ""
        status.value = ""

    def save_clicked(_):
        col = current_column()
        if col is None:
            status.value = "<span style='color:red'>No current column to save.</span>"
            return
        context = context_map.get(col, {}) if isinstance(context_map.get(col, {}), dict) else {}
        suggestion = _get_governance_ai_suggestion(ai_suggestion_map, col)
        row = {
            "environment_name": environment_name,
            "dataset_name": dataset_name,
            "table_name": table_name,
            "column_name": col,
            "metadata_table_key": build_metadata_table_key(environment_name, dataset_name, table_name),
            "metadata_column_key": build_metadata_column_key(environment_name, dataset_name, table_name, col),
            "business_context": context.get("business_context", ""),
            "business_context_notes": context.get("notes", "") or context.get("business_context_notes", ""),
            "confidentiality_label": confidentiality_dropdown.value,
            "personal_identifier_classification": personal_identifier_dropdown.value,
            "ai_suggested_personal_identifier_classification": suggestion.get("personal_identifier_classification"),
            "ai_suggested_reason": suggestion.get("reason"),
            "notes": notes_box.value.strip(),
            "status": "approved",
            "action_by": _resolve_action_by(),
            "action_ts": _now_utc_iso(),
            "source": "column_governance_widget",
        }
        COLUMN_GOVERNANCE_CONTEXT_FROM_WIDGET.append(row)
        state["history"].append({"action": "save", "saved_count": 1})
        state["index"] += 1
        load_current_column()

    def skip_clicked(_):
        if current_column() is None:
            status.value = "<span style='color:red'>No current column to skip.</span>"
            return
        state["history"].append({"action": "skip", "saved_count": 0})
        state["index"] += 1
        load_current_column()

    def undo_clicked(_):
        if not state["history"]:
            status.value = "<span style='color:orange'>Nothing to undo.</span>"
            return
        last = state["history"].pop()
        if last["saved_count"]:
            del COLUMN_GOVERNANCE_CONTEXT_FROM_WIDGET[-last["saved_count"]:]
        state["index"] = max(0, state["index"] - 1)
        load_current_column()

    save_button.on_click(save_clicked)
    skip_button.on_click(skip_clicked)
    undo_button.on_click(undo_clicked)

    ui = widgets.VBox([title, progress_label, column_summary, form_box, status], layout=widgets.Layout(border="1px solid #ddd", padding="12px", width="860px"))
    load_current_column()
    ipy_display(ui)
    return COLUMN_GOVERNANCE_CONTEXT_FROM_WIDGET


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
def write_metadata_rows(rows: list[dict], table_name: str, mode: str = "append"):
    """Simple notebook helper for writing list-of-dict metadata rows to a Spark table."""
    if not rows:
        raise ValueError(f"No rows supplied for {table_name}.")
    df = spark.createDataFrame(rows)
    df.write.mode(mode).saveAsTable(table_name)
    return df


def write_column_business_context(column_contexts, metadata_table: str = "METADATA_COLUMN_CONTEXT", mode: str = "append"):
    return write_metadata_rows(column_contexts, metadata_table, mode=mode)


def write_column_governance_context(governance_contexts, metadata_table: str = "METADATA_COLUMN_GOVERNANCE", mode: str = "append"):
    return write_metadata_rows(governance_contexts, metadata_table, mode=mode)


StatementMeta(, , -1, Waiting, , Waiting, True)

# Example flow


## 1. Create sample source data


In [ ]:
sample_rows = [
    {"event_count": 1, "message_id": "m001", "sender_email": "alice@example.edu", "status": "Delivered"},
    {"event_count": 1, "message_id": "m002", "sender_email": "bob@example.edu", "status": "Resolved"},
    {"event_count": -1, "message_id": "m003", "sender_email": None, "status": "Failed"},
    {"event_count": 2, "message_id": "m004", "sender_email": "carol@example.edu", "status": "Delivered"},
]

df_test = spark.createDataFrame(sample_rows)
display(df_test)


StatementMeta(, , -1, Waiting, , Waiting, True)

## 2. Set notebook variables


In [ ]:
ENVIRONMENT_NAME = "Sandbox"
DATASET_NAME = "Campus Email Logs"
TABLE_NAME = "EMAIL_LOGS"
TABLE_CONTEXT = "Campus email event metadata used for analytics and operational reporting."


StatementMeta(, , -1, Waiting, , Waiting, True)

## 3. Profile data


In [ ]:
profile_rows = profile_dataframe_for_dq(
    df=df_test,
    table_name=TABLE_NAME,
)

display(profile_rows)


StatementMeta(, , -1, Waiting, , Waiting, True)

## 4. AI first-pass business context


In [ ]:
ai_context_response_df = draft_business_context(
    profile_df=profile_rows,
    table_name=TABLE_NAME,
    table_context=TABLE_CONTEXT,
)

display(ai_context_response_df)


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
ai_context_suggestions = _extract_column_business_context_suggestions(ai_context_response_df)
#ai_context_suggestions


StatementMeta(, , -1, Waiting, , Waiting, True)

## 5. Human approves business context


In [ ]:
column_contexts = review_business_context(
    profile_df=profile_rows,
    table_name=TABLE_NAME,
    environment_name=ENVIRONMENT_NAME,
    dataset_name=DATASET_NAME,
    ai_suggested_contexts=ai_context_suggestions,
)


StatementMeta(, , -1, Waiting, , Waiting, True)

## 6. AI suggests DQ rules using approved business context


In [ ]:
responses = suggest_dq_rules_with_fabric_ai(
    profile_df=profile_rows,
    table_name=TABLE_NAME,
    column_contexts=COLUMN_BUSINESS_CONTEXT_FROM_WIDGET,
    output_col="response",
)

display(responses)


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
CANDIDATE_DQ_RULES = extract_candidate_rules_from_responses(
    response_df=responses,
    table_name=TABLE_NAME,
    environment_name=ENVIRONMENT_NAME,
    dataset_name=DATASET_NAME,
)

#CANDIDATE_DQ_RULES


StatementMeta(, , -1, Waiting, , Waiting, True)

## 7. Human approves DQ rules


In [ ]:
approved_rules, rejected_rules = review_dq_rules(
    CANDIDATE_DQ_RULES,
    table_name="EMAIL_LOGS",
    column_contexts=COLUMN_BUSINESS_CONTEXT_FROM_WIDGET,
)

StatementMeta(, , -1, Waiting, , Waiting, True)

## 8. Store and load approved DQ rules


In [ ]:
approved_rules_metadata_df = build_dq_rules_metadata_df(
    spark=spark,
    table_name=TABLE_NAME,
    environment_name=ENVIRONMENT_NAME,
    dataset_name=DATASET_NAME,
    approved_rules=APPROVED_RULES_FROM_WIDGET,
)

approved_rules_metadata_df.write.mode("append").saveAsTable("METADATA_DQ_RULES")
display(approved_rules_metadata_df)


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
APPROVED_DQ_RULES = load_latest_active_dq_rules_from_metadata(
    metadata_df=spark.table("METADATA_DQ_RULES"),
    table_name=TABLE_NAME,
    environment_name=ENVIRONMENT_NAME,
    dataset_name=DATASET_NAME,
)

#APPROVED_DQ_RULES


StatementMeta(, , -1, Waiting, , Waiting, True)

## 9. Enforce approved DQ rules


In [ ]:
dq_result = run_dq_rules(
    df=df_test,
    table_name=TABLE_NAME,
    rules=APPROVED_DQ_RULES,
)

display(dq_result)


StatementMeta(, , -1, Waiting, , Waiting, True)

## 10. Split valid and quarantine rows


In [ ]:
DQ_RUN_ID = str(uuid.uuid4())

df_valid, df_quarantine_rows, df_quarantine_failures = split_valid_quarantine_and_failures(
    df=df_test,
    rules=APPROVED_DQ_RULES,
    dq_run_id=DQ_RUN_ID,
    row_id_columns=["message_id"],
)

display(df_valid)
display(df_quarantine_rows)
display(df_quarantine_failures)


StatementMeta(, , -1, Waiting, , Waiting, True)

## 11. Optional: deactivate active DQ rules


In [ ]:
ACTIVE_DQ_RULES = load_latest_active_dq_rules_from_metadata(
    metadata_df=spark.table("METADATA_DQ_RULES"),
    table_name=TABLE_NAME,
    environment_name=ENVIRONMENT_NAME,
    dataset_name=DATASET_NAME,
)

deactivations, kept_rules = launch_grouped_rule_deactivation_widget(
    ACTIVE_DQ_RULES,
    table_name=TABLE_NAME,
)


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
deactivation_metadata_df = build_dq_rule_deactivation_metadata_df(
    spark=spark,
    table_name=TABLE_NAME,
    environment_name=ENVIRONMENT_NAME,
    dataset_name=DATASET_NAME,
    deactivations=RULE_DEACTIVATIONS_FROM_WIDGET,
)

deactivation_metadata_df.write.mode("append").saveAsTable("METADATA_DQ_RULES")
display(deactivation_metadata_df)


StatementMeta(, , -1, Waiting, , Waiting, True)

## 12. Governance AI uses approved business context


In [ ]:
ai_governance_response_df = suggest_personal_identifier_classifications(
    profile_df=profile_rows,
    table_name=TABLE_NAME,
    column_contexts=COLUMN_BUSINESS_CONTEXT_FROM_WIDGET,
)

display(ai_governance_response_df)


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
ai_governance_suggestions = extract_personal_identifier_suggestions(ai_governance_response_df)
ai_governance_suggestions


StatementMeta(, , -1, Waiting, , Waiting, True)

## 13. Human approves governance classification


In [ ]:
approved_governance = review_column_governance_context(
    profile_df=profile_rows,
    table_name=TABLE_NAME,
    environment_name=ENVIRONMENT_NAME,
    dataset_name=DATASET_NAME,
    column_contexts=COLUMN_BUSINESS_CONTEXT_FROM_WIDGET,
    governance_config=DEFAULT_GOVERNANCE_WIDGET_CONFIG,
    ai_suggestions=ai_governance_suggestions,
)


StatementMeta(, , -1, Waiting, , Waiting, True)

## 14. Optional: write column context and governance metadata


In [ ]:
# Optional metadata writes once the widgets are approved.
# These are separate metadata tables joined later by metadata_column_key.

column_context_df = write_column_business_context(COLUMN_BUSINESS_CONTEXT_FROM_WIDGET)
column_governance_df = write_column_governance_context(COLUMN_GOVERNANCE_CONTEXT_FROM_WIDGET)

display(column_context_df)
display(column_governance_df)


StatementMeta(, , -1, Waiting, , Waiting, True)